# Práctica SQL 02 — Caso 01: E-commerce Andino (preguntas)

10 preguntas básicas y 20 intermedias.

**Plataforma:** Databricks SQL · Delta Lake · Unity Catalog

## Descripción

Una empresa peruana de comercio electrónico vende productos de varias categorías mediante web, app y tiendas físicas. La gerencia necesita analizar clientes, pedidos, vendedores, ventas, descuentos, tiempos de entrega y devoluciones.

### Regla de negocio para los ejercicios

Salvo que la pregunta indique lo contrario, considera como **venta válida** todo pedido cuyo estado sea diferente de `cancelado`. El importe neto de una línea se calcula así:

```sql
cantidad * precio_unitario * (1 - descuento_pct / 100)
```

## Tablas

| Tabla | Columnas y tipos | Descripción |
|---|---|---|
| `clientes` | `cliente_id STRING`; `nombre_cliente STRING`; `ciudad STRING`; `region STRING`; `segmento STRING`; `fecha_registro DATE`; `email STRING` | Información demográfica y fecha de registro. |
| `vendedores` | `vendedor_id STRING`; `nombre_vendedor STRING`; `region STRING`; `fecha_ingreso DATE`; `meta_mensual DECIMAL(12,2)` | Región y meta mensual de cada vendedor. |
| `productos` | `producto_id STRING`; `nombre_producto STRING`; `categoria STRING`; `marca STRING`; `precio_lista DECIMAL(12,2)`; `costo_unitario DECIMAL(12,2)`; `activo BOOLEAN` | Catálogo, precios y costos. |
| `pedidos` | `pedido_id STRING`; `cliente_id STRING`; `vendedor_id STRING`; `fecha_pedido DATE`; `canal STRING`; `metodo_pago STRING`; `estado STRING`; `ciudad_envio STRING`; `fecha_entrega DATE` *(admite NULL)* | Cabecera del pedido, canal, estado y entrega. |
| `detalle_pedidos` | `pedido_id STRING`; `linea_id INT`; `producto_id STRING`; `cantidad INT`; `precio_unitario DECIMAL(12,2)`; `descuento_pct DECIMAL(5,2)` | Productos, cantidades, precios y descuentos. |
| `devoluciones` | `devolucion_id STRING`; `pedido_id STRING`; `producto_id STRING`; `fecha_devolucion DATE`; `cantidad_devuelta INT`; `motivo STRING`; `monto_reembolso DECIMAL(12,2)` | Productos devueltos y montos reembolsados. |

## Relaciones

- `clientes.cliente_id` → `pedidos.cliente_id` (1 a N)
- `vendedores.vendedor_id` → `pedidos.vendedor_id` (1 a N)
- `pedidos.pedido_id` → `detalle_pedidos.pedido_id` (1 a N)
- `productos.producto_id` → `detalle_pedidos.producto_id` (1 a N)
- `pedidos.pedido_id` → `devoluciones.pedido_id` (1 a N)
- `productos.producto_id` → `devoluciones.producto_id` (1 a N)

## Consideraciones

- Los archivos TXT usan formato CSV con encabezado y codificación UTF-8.
- Las fechas usan el formato `YYYY-MM-DD`.
- Los campos vacíos representan `NULL`, principalmente `fecha_entrega` (pedidos aún no entregados).
- Los pedidos cancelados conservan sus líneas para permitir analizar intentos de compra, pero **no** se consideran ventas válidas.

---
## PARTE 0 — Preparación del entorno (DDL y carga)

> Si ya creaste y cargaste las tablas de este caso, salta directamente a la **PARTE 1**.

### 0.1 Esquemas y volumen

| Objeto | Nombre | Para qué sirve |
|---|---|---|
| Esquema | `workspace.sql_practice_raw` | Contiene el volumen con los archivos originales. |
| Volumen | `workspace.sql_practice_raw.datasets` | Almacena los `.txt` de ambos casos, cada uno en su carpeta. |
| Esquema | `workspace.sql_practice_ecommerce` | Contiene las 6 tablas de este caso. |

En Databricks Free Edition el catálogo por defecto es `workspace`. Si el tuyo se llama `main` u otro nombre, reemplaza `workspace` en todo el notebook, incluidas las rutas `/Volumes/workspace/...`.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.sql_practice_raw;
CREATE VOLUME IF NOT EXISTS workspace.sql_practice_raw.datasets;

CREATE SCHEMA IF NOT EXISTS workspace.sql_practice_ecommerce;

### 0.2 Subir los archivos al volumen

1. En **Catalog Explorer** abre el volumen `workspace` → `sql_practice_raw` → **Volumes** → `datasets`.
2. Crea la carpeta **`caso_01_ecommerce_andino`** (si aún no existe).
3. Dentro de esa carpeta, sube los 6 archivos de `datos_txt/`: `clientes.txt`, `vendedores.txt`, `productos.txt`, `pedidos.txt`, `detalle_pedidos.txt`, `devoluciones.txt`.
4. Ejecuta la siguiente celda: deben aparecer los 6 archivos.

In [0]:
%sql
LIST '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/';

path,name,size,modification_time
/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/clientes.txt,clientes.txt,9738,1792881940000
/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/detalle_pedidos.txt,detalle_pedidos.txt,31341,1792881940000
/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/devoluciones.txt,devoluciones.txt,3774,1792881940000
/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/pedidos.txt,pedidos.txt,24461,1792881940000
/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/productos.txt,productos.txt,2680,1792881940000
/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/vendedores.txt,vendedores.txt,617,1792881940000


### 0.3 Eliminar tablas previas

Permite volver a ejecutar la preparación desde cero. Las tablas se eliminan en **orden inverso a sus dependencias**: primero las hijas y luego las padres.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA sql_practice_ecommerce;

DROP TABLE IF EXISTS devoluciones;
DROP TABLE IF EXISTS detalle_pedidos;
DROP TABLE IF EXISTS pedidos;
DROP TABLE IF EXISTS productos;
DROP TABLE IF EXISTS vendedores;
DROP TABLE IF EXISTS clientes;

### 0.4 Crear las tablas (DDL)

- Todas las columnas son `NOT NULL`, salvo las indicadas como *admite NULL*. Delta Lake **sí** valida esta restricción al insertar.
- `PRIMARY KEY` y `FOREIGN KEY` en Unity Catalog son restricciones **informativas**: documentan el modelo, pero **no se validan** al insertar. Por eso en el paso 0.6 verificamos la integridad con consultas.
- Las tablas se crean de **padres a hijas**: una clave foránea solo puede apuntar a una tabla que ya tenga su clave primaria.

#### `clientes`

In [0]:
%sql
CREATE TABLE clientes (
  cliente_id      STRING  NOT NULL COMMENT 'Identificador del cliente',
  nombre_cliente  STRING  NOT NULL COMMENT 'Nombre completo',
  ciudad          STRING  NOT NULL COMMENT 'Ciudad de residencia',
  region          STRING  NOT NULL COMMENT 'Región del país',
  segmento        STRING  NOT NULL COMMENT 'Segmento del cliente',
  fecha_registro  DATE    NOT NULL COMMENT 'Fecha de registro',
  email           STRING  NOT NULL COMMENT 'Correo electrónico (ficticio)',
  CONSTRAINT pk_clientes PRIMARY KEY (cliente_id)
)
COMMENT 'Clientes de la tienda online.';

#### `vendedores`

In [0]:
%sql
CREATE TABLE vendedores (
  vendedor_id      STRING         NOT NULL COMMENT 'Identificador del vendedor',
  nombre_vendedor  STRING         NOT NULL COMMENT 'Nombre completo',
  region           STRING         NOT NULL COMMENT 'Región asignada',
  fecha_ingreso    DATE           NOT NULL COMMENT 'Fecha de ingreso a la empresa',
  meta_mensual     DECIMAL(12,2)  NOT NULL COMMENT 'Meta de ventas mensual en soles',
  CONSTRAINT pk_vendedores PRIMARY KEY (vendedor_id)
)
COMMENT 'Vendedores y su meta mensual.';

#### `productos`

In [0]:
%sql
CREATE TABLE productos (
  producto_id      STRING         NOT NULL COMMENT 'Identificador del producto',
  nombre_producto  STRING         NOT NULL COMMENT 'Nombre del producto',
  categoria        STRING         NOT NULL COMMENT 'Categoría del producto',
  marca            STRING         NOT NULL COMMENT 'Marca',
  precio_lista     DECIMAL(12,2)  NOT NULL COMMENT 'Precio de lista en soles',
  costo_unitario   DECIMAL(12,2)  NOT NULL COMMENT 'Costo unitario en soles',
  activo           BOOLEAN        NOT NULL COMMENT 'Indica si el producto está activo',
  CONSTRAINT pk_productos PRIMARY KEY (producto_id)
)
COMMENT 'Catálogo de productos.';

#### `pedidos`

In [0]:
%sql
CREATE TABLE pedidos (
  pedido_id      STRING  NOT NULL COMMENT 'Identificador del pedido',
  cliente_id     STRING  NOT NULL COMMENT 'Cliente que realiza el pedido',
  vendedor_id    STRING  NOT NULL COMMENT 'Vendedor responsable',
  fecha_pedido   DATE    NOT NULL COMMENT 'Fecha del pedido',
  canal          STRING  NOT NULL COMMENT 'Canal de venta',
  metodo_pago    STRING  NOT NULL COMMENT 'Método de pago',
  estado         STRING  NOT NULL COMMENT 'Estado del pedido',
  ciudad_envio   STRING  NOT NULL COMMENT 'Ciudad de envío',
  fecha_entrega  DATE             COMMENT 'Fecha de entrega (NULL si aún no se entrega)',
  CONSTRAINT pk_pedidos PRIMARY KEY (pedido_id),
  CONSTRAINT fk_pedidos_clientes FOREIGN KEY (cliente_id) REFERENCES clientes (cliente_id),
  CONSTRAINT fk_pedidos_vendedores FOREIGN KEY (vendedor_id) REFERENCES vendedores (vendedor_id)
)
COMMENT 'Cabecera de pedidos.';

#### `detalle_pedidos`

In [0]:
%sql
CREATE TABLE detalle_pedidos (
  pedido_id        STRING         NOT NULL COMMENT 'Pedido al que pertenece la línea',
  linea_id         INT            NOT NULL COMMENT 'Número de línea dentro del pedido',
  producto_id      STRING         NOT NULL COMMENT 'Producto vendido',
  cantidad         INT            NOT NULL COMMENT 'Unidades vendidas',
  precio_unitario  DECIMAL(12,2)  NOT NULL COMMENT 'Precio unitario aplicado en soles',
  descuento_pct    DECIMAL(5,2)   NOT NULL COMMENT 'Descuento en porcentaje (0 a 100)',
  CONSTRAINT pk_detalle_pedidos PRIMARY KEY (pedido_id, linea_id),
  CONSTRAINT fk_detalle_pedidos_pedidos FOREIGN KEY (pedido_id) REFERENCES pedidos (pedido_id),
  CONSTRAINT fk_detalle_pedidos_productos FOREIGN KEY (producto_id) REFERENCES productos (producto_id)
)
COMMENT 'Líneas de cada pedido.';

#### `devoluciones`

In [0]:
%sql
CREATE TABLE devoluciones (
  devolucion_id      STRING         NOT NULL COMMENT 'Identificador de la devolución',
  pedido_id          STRING         NOT NULL COMMENT 'Pedido de origen',
  producto_id        STRING         NOT NULL COMMENT 'Producto devuelto',
  fecha_devolucion   DATE           NOT NULL COMMENT 'Fecha de la devolución',
  cantidad_devuelta  INT            NOT NULL COMMENT 'Unidades devueltas',
  motivo             STRING         NOT NULL COMMENT 'Motivo de la devolución',
  monto_reembolso    DECIMAL(12,2)  NOT NULL COMMENT 'Monto reembolsado en soles',
  CONSTRAINT pk_devoluciones PRIMARY KEY (devolucion_id),
  CONSTRAINT fk_devoluciones_pedidos FOREIGN KEY (pedido_id) REFERENCES pedidos (pedido_id),
  CONSTRAINT fk_devoluciones_productos FOREIGN KEY (producto_id) REFERENCES productos (producto_id)
)
COMMENT 'Productos devueltos y montos reembolsados.';

### 0.5 Cargar los datos

Usamos `read_files` para leer cada `.txt` del volumen:

- `format => 'csv'` y `header => true`: texto delimitado por comas con encabezado.
- `schema => '...'`: define el tipo de cada columna en lugar de dejar que Databricks lo infiera. Los campos vacíos se cargan como `NULL`.
- `mode => 'FAILFAST'`: si alguna fila no se puede convertir al tipo indicado, la carga falla en lugar de dejar datos incorrectos.

#### `clientes`

In [0]:
%sql
INSERT INTO clientes
SELECT
  cliente_id,
  nombre_cliente,
  ciudad,
  region,
  segmento,
  fecha_registro,
  email
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/clientes.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'cliente_id STRING, nombre_cliente STRING, ciudad STRING, region STRING, segmento STRING, fecha_registro DATE, email STRING'
);

SELECT * FROM clientes LIMIT 5;

cliente_id,nombre_cliente,ciudad,region,segmento,fecha_registro,email
C0001,Renzo Reyes,Cusco,Sur,Persona,2024-03-06,renzo.reyes1@example.com
C0002,Natalia Aguilar,Piura,Norte,Persona,2024-12-04,natalia.aguilar2@example.com
C0003,Hugo Núñez,Cajamarca,Norte,Empresa,2024-10-12,hugo.nunez3@example.com
C0004,Carla Mendoza,Cusco,Sur,Persona,2025-08-26,carla.mendoza4@example.com
C0005,Carlos Silva,Puno,Sur,Persona,2024-01-19,carlos.silva5@example.com


#### `vendedores`

In [0]:
%sql
INSERT INTO vendedores
SELECT
  vendedor_id,
  nombre_vendedor,
  region,
  fecha_ingreso,
  meta_mensual
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/vendedores.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'vendedor_id STRING, nombre_vendedor STRING, region STRING, fecha_ingreso DATE, meta_mensual DECIMAL(12,2)'
);

SELECT * FROM vendedores LIMIT 5;

vendedor_id,nombre_vendedor,region,fecha_ingreso,meta_mensual
V001,José Cárdenas,Costa,2022-10-29,25000.00
V002,José Salazar,Norte,2024-07-29,15000.00
V003,Fernando Torres,Centro,2023-06-09,30000.00
V004,Camila Herrera,Sur,2023-08-16,18000.00
V005,Paola Silva,Costa,2025-01-21,30000.00


#### `productos`

In [0]:
%sql
INSERT INTO productos
SELECT
  producto_id,
  nombre_producto,
  categoria,
  marca,
  precio_lista,
  costo_unitario,
  activo
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/productos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'producto_id STRING, nombre_producto STRING, categoria STRING, marca STRING, precio_lista DECIMAL(12,2), costo_unitario DECIMAL(12,2), activo BOOLEAN'
);

SELECT * FROM productos LIMIT 5;

producto_id,nombre_producto,categoria,marca,precio_lista,costo_unitario,activo
P0001,Laptop Nova 14,Tecnología,NovaTech,2130.00,1295.55,true
P0002,Mouse Ergonómico,Tecnología,ClickPro,1095.00,590.93,true
P0003,Teclado Mecánico,Tecnología,KeyMax,2220.00,1335.93,true
P0004,Audífonos Bluetooth,Tecnología,SoundGo,735.00,518.89,true
P0005,Monitor 24 pulgadas,Tecnología,ViewPlus,2715.00,1746.64,false


#### `pedidos`

In [0]:
%sql
INSERT INTO pedidos
SELECT
  pedido_id,
  cliente_id,
  vendedor_id,
  fecha_pedido,
  canal,
  metodo_pago,
  estado,
  ciudad_envio,
  fecha_entrega
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/pedidos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'pedido_id STRING, cliente_id STRING, vendedor_id STRING, fecha_pedido DATE, canal STRING, metodo_pago STRING, estado STRING, ciudad_envio STRING, fecha_entrega DATE'
);

SELECT * FROM pedidos LIMIT 5;

pedido_id,cliente_id,vendedor_id,fecha_pedido,canal,metodo_pago,estado,ciudad_envio,fecha_entrega
O00001,C0037,V002,2025-08-13,App,Tarjeta,entregado,Cajamarca,2025-08-24
O00002,C0092,V007,2026-06-19,App,Plin,pendiente,Ayacucho,null
O00003,C0035,V012,2025-09-24,Tienda,Tarjeta,entregado,Puno,2025-09-30
O00004,C0117,V008,2026-05-11,Web,Tarjeta,entregado,Cusco,2026-05-16
O00005,C0057,V010,2025-09-16,App,Efectivo,entregado,Trujillo,2025-09-23


#### `detalle_pedidos`

In [0]:
%sql
INSERT INTO detalle_pedidos
SELECT
  pedido_id,
  linea_id,
  producto_id,
  cantidad,
  precio_unitario,
  descuento_pct
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/detalle_pedidos.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'pedido_id STRING, linea_id INT, producto_id STRING, cantidad INT, precio_unitario DECIMAL(12,2), descuento_pct DECIMAL(5,2)'
);

SELECT * FROM detalle_pedidos LIMIT 5;

pedido_id,linea_id,producto_id,cantidad,precio_unitario,descuento_pct
O00001,1,P0034,1,136.57,10.00
O00001,2,P0024,1,88.29,15.00
O00001,3,P0012,1,167.53,25.00
O00001,4,P0031,2,110.51,0.00
O00002,1,P0004,2,708.06,10.00


#### `devoluciones`

In [0]:
%sql
INSERT INTO devoluciones
SELECT
  devolucion_id,
  pedido_id,
  producto_id,
  fecha_devolucion,
  cantidad_devuelta,
  motivo,
  monto_reembolso
FROM read_files(
  '/Volumes/workspace/sql_practice_raw/datasets/caso_01_ecommerce_andino/devoluciones.txt',
  format => 'csv',
  header => true,
  mode => 'FAILFAST',
  schema => 'devolucion_id STRING, pedido_id STRING, producto_id STRING, fecha_devolucion DATE, cantidad_devuelta INT, motivo STRING, monto_reembolso DECIMAL(12,2)'
);

SELECT * FROM devoluciones LIMIT 5;

devolucion_id,pedido_id,producto_id,fecha_devolucion,cantidad_devuelta,motivo,monto_reembolso
R0001,O00001,P0034,2025-09-04,1,Talla incorrecta,122.91
R0002,O00006,P0039,2026-01-26,1,Talla incorrecta,780.62
R0003,O00012,P0015,2025-12-03,3,No era lo esperado,800.70
R0004,O00029,P0005,2025-11-17,1,Talla incorrecta,2175.86
R0005,O00055,P0006,2025-08-23,1,Talla incorrecta,2018.57


### 0.6 Validación

Todas las tablas deben mostrar `estado = OK`.

In [0]:
%sql
WITH conteos AS (
  SELECT 'clientes' AS tabla, COUNT(*) AS filas, 120 AS filas_esperadas FROM clientes
  UNION ALL
  SELECT 'vendedores' AS tabla, COUNT(*) AS filas, 12 AS filas_esperadas FROM vendedores
  UNION ALL
  SELECT 'productos' AS tabla, COUNT(*) AS filas, 45 AS filas_esperadas FROM productos
  UNION ALL
  SELECT 'pedidos' AS tabla, COUNT(*) AS filas, 360 AS filas_esperadas FROM pedidos
  UNION ALL
  SELECT 'detalle_pedidos' AS tabla, COUNT(*) AS filas, 1030 AS filas_esperadas FROM detalle_pedidos
  UNION ALL
  SELECT 'devoluciones' AS tabla, COUNT(*) AS filas, 63 AS filas_esperadas FROM devoluciones
)
SELECT
  tabla,
  filas,
  filas_esperadas,
  IF(filas = filas_esperadas, 'OK', 'REVISAR') AS estado
FROM conteos;

tabla,filas,filas_esperadas,estado
clientes,120,120,OK
vendedores,12,12,OK
productos,45,45,OK
pedidos,360,360,OK
detalle_pedidos,1030,1030,OK
devoluciones,63,63,OK


Integridad referencial: como las claves foráneas son informativas, comprobamos que no existan filas hijas que apunten a un padre inexistente. Todas las relaciones deben mostrar `huerfanos = 0`.

In [0]:
%sql
SELECT 'pedidos.cliente_id -> clientes' AS relacion, COUNT(*) AS huerfanos
FROM pedidos AS h
LEFT JOIN clientes AS p ON h.cliente_id = p.cliente_id
WHERE p.cliente_id IS NULL
UNION ALL
SELECT 'pedidos.vendedor_id -> vendedores' AS relacion, COUNT(*) AS huerfanos
FROM pedidos AS h
LEFT JOIN vendedores AS p ON h.vendedor_id = p.vendedor_id
WHERE p.vendedor_id IS NULL
UNION ALL
SELECT 'detalle_pedidos.pedido_id -> pedidos' AS relacion, COUNT(*) AS huerfanos
FROM detalle_pedidos AS h
LEFT JOIN pedidos AS p ON h.pedido_id = p.pedido_id
WHERE p.pedido_id IS NULL
UNION ALL
SELECT 'detalle_pedidos.producto_id -> productos' AS relacion, COUNT(*) AS huerfanos
FROM detalle_pedidos AS h
LEFT JOIN productos AS p ON h.producto_id = p.producto_id
WHERE p.producto_id IS NULL
UNION ALL
SELECT 'devoluciones.pedido_id -> pedidos' AS relacion, COUNT(*) AS huerfanos
FROM devoluciones AS h
LEFT JOIN pedidos AS p ON h.pedido_id = p.pedido_id
WHERE p.pedido_id IS NULL
UNION ALL
SELECT 'devoluciones.producto_id -> productos' AS relacion, COUNT(*) AS huerfanos
FROM devoluciones AS h
LEFT JOIN productos AS p ON h.producto_id = p.producto_id
WHERE p.producto_id IS NULL;

relacion,huerfanos
pedidos.cliente_id -> clientes,0
pedidos.vendedor_id -> vendedores,0
detalle_pedidos.pedido_id -> pedidos,0
detalle_pedidos.producto_id -> productos,0
devoluciones.pedido_id -> pedidos,0
devoluciones.producto_id -> productos,0


---
## Configuración para resolver las preguntas

Ejecuta esta celda cada vez que abras el notebook: las consultas usan nombres de tablas sin catálogo ni esquema.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA sql_practice_ecommerce;

SHOW TABLES;

database,tableName,isTemporary
sql_practice_ecommerce,clientes,false
sql_practice_ecommerce,detalle_pedidos,false
sql_practice_ecommerce,devoluciones,false
sql_practice_ecommerce,pedidos,false
sql_practice_ecommerce,productos,false
sql_practice_ecommerce,vendedores,false


---
## PARTE 1 — Preguntas básicas (1-10)

### 1. Todos los clientes

Muestra todos los registros de la tabla `clientes`.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT *
FROM clientes;


cliente_id,nombre_cliente,ciudad,region,segmento,fecha_registro,email
C0001,Renzo Reyes,Cusco,Sur,Persona,2024-03-06,renzo.reyes1@example.com
C0002,Natalia Aguilar,Piura,Norte,Persona,2024-12-04,natalia.aguilar2@example.com
C0003,Hugo Núñez,Cajamarca,Norte,Empresa,2024-10-12,hugo.nunez3@example.com
C0004,Carla Mendoza,Cusco,Sur,Persona,2025-08-26,carla.mendoza4@example.com
C0005,Carlos Silva,Puno,Sur,Persona,2024-01-19,carlos.silva5@example.com
C0006,Bruno Ortega,Ica,Costa,Persona,2025-11-07,bruno.ortega6@example.com
C0007,Daniela Navarro,Huancayo,Centro,Persona,2024-03-02,daniela.navarro7@example.com
C0008,Gabriela Acosta,Arequipa,Sur,Persona,2025-10-27,gabriela.acosta8@example.com
C0009,Miguel Paredes,Lima,Costa,Persona,2025-11-10,miguel.paredes9@example.com
C0010,Jorge Reyes,Tacna,Sur,Pyme,2025-08-26,jorge.reyes10@example.com


### 2. Productos activos

Lista el ID, nombre, categoría y precio de los productos activos.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
  producto_id,
  nombre_producto,
  categoria,
  precio_lista
FROM productos
WHERE activo = TRUE;

producto_id,nombre_producto,categoria,precio_lista
P0001,Laptop Nova 14,Tecnología,2130.00
P0002,Mouse Ergonómico,Tecnología,1095.00
P0003,Teclado Mecánico,Tecnología,2220.00
P0004,Audífonos Bluetooth,Tecnología,735.00
P0006,Webcam Full HD,Tecnología,2135.00
P0007,SSD Externo 1TB,Tecnología,205.00
P0008,Smartwatch Active,Tecnología,1855.00
P0010,Cafetera Digital,Hogar,145.00
P0011,Freidora de Aire,Hogar,690.00
P0012,Juego de Sábanas,Hogar,165.00


### 3. Pedidos entregados

Muestra los pedidos cuyo estado sea `entregado`.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT *
FROM pedidos
WHERE estado = 'entregado';

pedido_id,cliente_id,vendedor_id,fecha_pedido,canal,metodo_pago,estado,ciudad_envio,fecha_entrega
O00001,C0037,V002,2025-08-13,App,Tarjeta,entregado,Cajamarca,2025-08-24
O00003,C0035,V012,2025-09-24,Tienda,Tarjeta,entregado,Puno,2025-09-30
O00004,C0117,V008,2026-05-11,Web,Tarjeta,entregado,Cusco,2026-05-16
O00005,C0057,V010,2025-09-16,App,Efectivo,entregado,Trujillo,2025-09-23
O00006,C0006,V001,2026-01-07,Web,Plin,entregado,Ica,2026-01-14
O00007,C0039,V010,2025-12-08,Web,Tarjeta,entregado,Trujillo,2025-12-18
O00009,C0099,V007,2026-01-11,Web,Transferencia,entregado,Ayacucho,2026-01-14
O00010,C0037,V006,2025-02-13,App,Tarjeta,entregado,Cajamarca,2025-02-15
O00012,C0111,V007,2025-11-28,Web,Transferencia,entregado,Huancayo,2025-12-01
O00013,C0034,V006,2025-02-24,App,Yape,entregado,Trujillo,2025-02-27


### 4. Ciudades únicas de clientes

Obtén la lista de ciudades únicas de los clientes, ordenadas alfabéticamente.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT DISTINCT ciudad
FROM clientes
ORDER BY ciudad ASC;

ciudad
Arequipa
Ayacucho
Cajamarca
Chiclayo
Cusco
Huancayo
Ica
Lima
Piura
Puno


### 5. Diez productos con mayor precio

Muestra los 10 productos con mayor precio de lista.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT *
FROM productos
ORDER BY precio_lista DESC
LIMIT 10;

producto_id,nombre_producto,categoria,marca,precio_lista,costo_unitario,activo
P0005,Monitor 24 pulgadas,Tecnología,ViewPlus,2715.00,1746.64,false
P0003,Teclado Mecánico,Tecnología,KeyMax,2220.00,1335.93,true
P0006,Webcam Full HD,Tecnología,VisionX,2135.00,1434.37,true
P0001,Laptop Nova 14,Tecnología,NovaTech,2130.00,1295.55,true
P0008,Smartwatch Active,Tecnología,Pulse,1855.00,959.23,true
P0002,Mouse Ergonómico,Tecnología,ClickPro,1095.00,590.93,true
P0044,Pizarra Magnética,Oficina,IdeaBoard,940.00,590.83,true
P0039,Hub USB-C,Oficina,LinkPro,940.00,607.02,true
P0040,Soporte para Laptop,Oficina,DeskUp,890.00,582.23,true
P0004,Audífonos Bluetooth,Tecnología,SoundGo,735.00,518.89,true


### 6. Cantidad total de pedidos

Cuenta cuántos pedidos existen en total.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT COUNT(*) AS cantidad_total_pedidos
FROM pedidos;

cantidad_total_pedidos
360


### 7. Precio promedio de los productos

Calcula el precio de lista promedio de todos los productos.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT AVG(precio_lista) AS precio_promedio
FROM productos;

precio_promedio
543.222222


### 8. Pedidos de enero de 2026

Muestra los pedidos realizados durante enero de 2026.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT *
FROM pedidos
WHERE fecha_pedido >= '2026-01-01'
  AND fecha_pedido < '2026-02-01'
ORDER BY fecha_pedido;

pedido_id,cliente_id,vendedor_id,fecha_pedido,canal,metodo_pago,estado,ciudad_envio,fecha_entrega
O00168,C0041,V006,2026-01-01,App,Efectivo,entregado,Chiclayo,2026-01-05
O00284,C0117,V004,2026-01-01,Web,Yape,entregado,Cusco,2026-01-08
O00066,C0091,V002,2026-01-02,Web,Yape,entregado,Chiclayo,2026-01-09
O00314,C0012,V004,2026-01-02,App,Tarjeta,enviado,Arequipa,null
O00064,C0083,V002,2026-01-02,Web,Transferencia,pendiente,Piura,null
O00205,C0011,V009,2026-01-03,Web,Plin,enviado,Lima,null
O00173,C0004,V004,2026-01-04,Web,Tarjeta,cancelado,Cusco,null
O00279,C0084,V011,2026-01-07,App,Plin,entregado,Ayacucho,2026-01-11
O00006,C0006,V001,2026-01-07,Web,Plin,entregado,Ica,2026-01-14
O00244,C0040,V010,2026-01-09,App,Transferencia,entregado,Piura,2026-01-10


### 9. Clientes por segmento

Cuenta cuántos clientes existen por segmento.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
  segmento,
  COUNT(*) AS cantidad_clientes
FROM clientes
GROUP BY segmento
ORDER BY segmento;

segmento,cantidad_clientes
Empresa,10
Persona,76
Pyme,34


### 10. Importe neto por línea

Muestra cada línea de pedido incluyendo una columna calculada `importe_neto_linea`.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
  *,
  cantidad * precio_unitario *
    (1 - descuento_pct / 100.0) AS importe_neto_linea
FROM detalle_pedidos;

pedido_id,linea_id,producto_id,cantidad,precio_unitario,descuento_pct,importe_neto_linea
O00001,1,P0034,1,136.57,10.00,122.913000000
O00001,2,P0024,1,88.29,15.00,75.046500000
O00001,3,P0012,1,167.53,25.00,125.647500000
O00001,4,P0031,2,110.51,0.00,221.020000000
O00002,1,P0004,2,708.06,10.00,1274.508000000
O00003,1,P0008,2,1807.73,0.00,3615.460000000
O00003,2,P0029,3,101.84,20.00,244.416000000
O00003,3,P0002,1,1117.27,0.00,1117.270000000
O00003,4,P0030,1,150.76,25.00,113.070000000
O00004,1,P0040,1,920.72,5.00,874.684000000


---
## PARTE 2 — Preguntas intermedias (11-30)

### 11. Clientes con cantidad de pedidos, incluyendo clientes sin pedidos

Muestra cada cliente con su cantidad total de pedidos, incluyendo clientes sin pedidos.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    c.cliente_id,
    c.nombre_cliente,
    COUNT(p.pedido_id) AS cantidad_pedidos
FROM clientes c
LEFT JOIN pedidos p
    ON c.cliente_id = p.cliente_id
GROUP BY c.cliente_id, c.nombre_cliente
ORDER BY c.nombre_cliente;

cliente_id,nombre_cliente,cantidad_pedidos
C0034,Alejandra Aguilar,6
C0112,Alejandra Campos,4
C0030,Alejandra Chávez,2
C0111,Alejandra Flores,4
C0094,Ana Campos,4
C0085,Ana Cruz,4
C0082,Ana Sánchez,2
C0049,Ana Torres,2
C0027,Ana Valdez,5
C0088,Andrea Castro,0


### 12. Ventas netas válidas por categoría

Calcula las ventas netas válidas por categoría de producto.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    pr.categoria,
    SUM(dp.cantidad * dp.precio_unitario *
        (1 - dp.descuento_pct / 100.0)) AS ventas_netas
FROM detalle_pedidos dp
JOIN pedidos p
    ON dp.pedido_id = p.pedido_id
JOIN productos pr
    ON dp.producto_id = pr.producto_id
WHERE p.estado <> 'cancelado'
GROUP BY pr.categoria
ORDER BY ventas_netas DESC;

categoria,ventas_netas
Tecnología,484789.875500000
Oficina,125504.248000000
Hogar,107762.248500000
Deportes,46057.548500000
Belleza,35519.956000000
Moda,24510.648000000


### 13. Ticket promedio de pedidos entregados

Calcula el ticket promedio de los pedidos entregados. Primero suma cada pedido y luego promedia esos totales.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT AVG(total_pedido) AS ticket_promedio
FROM (
    SELECT
        p.pedido_id,
        SUM(dp.cantidad * dp.precio_unitario *
            (1 - dp.descuento_pct / 100.0)) AS total_pedido
    FROM pedidos p
    JOIN detalle_pedidos dp
        ON p.pedido_id = dp.pedido_id
    WHERE p.estado = 'entregado'
    GROUP BY p.pedido_id
) AS totales;

ticket_promedio
2571.0531428571429


### 14. Pedidos válidos y ventas por mes

Obtén la cantidad de pedidos válidos y las ventas netas por mes.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    DATE_FORMAT(p.fecha_pedido, 'yyyy-MM') AS mes,
    COUNT(DISTINCT p.pedido_id) AS cantidad_pedidos,
    SUM(dp.cantidad * dp.precio_unitario *
        (1 - dp.descuento_pct / 100.0)) AS ventas_netas
FROM pedidos p
JOIN detalle_pedidos dp
    ON p.pedido_id = dp.pedido_id
WHERE p.estado <> 'cancelado'
GROUP BY DATE_FORMAT(p.fecha_pedido, 'yyyy-MM')
ORDER BY mes;

mes,cantidad_pedidos,ventas_netas
2025-01,10,30543.700500000
2025-02,11,37437.390500000
2025-03,6,14321.272000000
2025-04,10,29019.871000000
2025-05,8,16668.402500000
2025-06,9,21692.889000000
2025-07,17,48426.429500000
2025-08,18,45525.978500000
2025-09,16,46879.126000000
2025-10,18,52283.788000000


### 15. Top 10 clientes por gasto neto

Identifica los 10 clientes con mayor gasto neto en pedidos válidos.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    c.cliente_id,
    c.nombre_cliente,
    SUM(dp.cantidad * dp.precio_unitario *
        (1 - dp.descuento_pct / 100.0)) AS gasto_neto
FROM clientes c
JOIN pedidos p
    ON c.cliente_id = p.cliente_id
JOIN detalle_pedidos dp
    ON p.pedido_id = dp.pedido_id
WHERE p.estado <> 'cancelado'
GROUP BY c.cliente_id, c.nombre_cliente
ORDER BY gasto_neto DESC
LIMIT 10;

cliente_id,nombre_cliente,gasto_neto
C0027,Ana Valdez,31382.797000000
C0090,Camila Medina,29452.630000000
C0019,Daniela Peña,26312.579000000
C0096,Gabriela Silva,23669.585000000
C0098,María Acosta,22783.139500000
C0011,Álvaro Silva,21690.868500000
C0034,Alejandra Aguilar,20064.624500000
C0101,Marco Campos,19790.046000000
C0087,Carla Ramírez,16024.258500000
C0076,Mateo Vargas,15415.046000000


### 16. Productos nunca vendidos

Lista los productos que nunca aparecen en `detalle_pedidos`.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    pr.producto_id,
    pr.nombre_producto
FROM productos pr
LEFT JOIN detalle_pedidos dp
    ON pr.producto_id = dp.producto_id
WHERE dp.producto_id IS NULL;

producto_id,nombre_producto
P0043,Calculadora Científica
P0044,Pizarra Magnética
P0045,Archivador A4


### 17. Clientes sin pedidos

Lista los clientes que nunca realizaron un pedido.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    c.cliente_id,
    c.nombre_cliente
FROM clientes c
LEFT JOIN pedidos p
    ON c.cliente_id = p.cliente_id
WHERE p.pedido_id IS NULL;

cliente_id,nombre_cliente
C0029,Carla Reyes
C0051,Marco Mendoza
C0055,Jorge Rojas
C0066,Sofía Sánchez
C0075,María Espinoza
C0081,Fernando Herrera
C0088,Andrea Castro
C0113,Natalia Cruz


### 18. Promedio de días de entrega por canal

Calcula el promedio de días de entrega por canal para pedidos entregados.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    canal,
    AVG(DATEDIFF(fecha_entrega, fecha_pedido)) AS promedio_dias_entrega
FROM pedidos
WHERE estado = 'entregado'
GROUP BY canal
ORDER BY canal;

canal,promedio_dias_entrega
App,5.958762886597938
Tienda,5.421052631578948
Web,6.591836734693878


### 19. Unidades vendidas, devueltas y porcentaje de devolución por categoría

Calcula por categoría: unidades vendidas, unidades devueltas y porcentaje de devolución sobre unidades vendidas. Considera ventas válidas.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    v.categoria,
    v.unidades_vendidas,
    COALESCE(d.unidades_devueltas, 0) AS unidades_devueltas,
    COALESCE(d.unidades_devueltas, 0) * 100.0 /
        v.unidades_vendidas AS porcentaje_devolucion
FROM (
    SELECT
        pr.categoria,
        SUM(dp.cantidad) AS unidades_vendidas
    FROM detalle_pedidos dp
    JOIN pedidos p ON dp.pedido_id = p.pedido_id
    JOIN productos pr ON dp.producto_id = pr.producto_id
    WHERE p.estado <> 'cancelado'
    GROUP BY pr.categoria
) v
LEFT JOIN (
    SELECT
        pr.categoria,
        SUM(d.cantidad_devuelta) AS unidades_devueltas
    FROM devoluciones d
    JOIN pedidos p ON d.pedido_id = p.pedido_id
    JOIN productos pr ON d.producto_id = pr.producto_id
    WHERE p.estado <> 'cancelado'
    GROUP BY pr.categoria
) d
    ON v.categoria = d.categoria
ORDER BY v.categoria;

categoria,unidades_vendidas,unidades_devueltas,porcentaje_devolucion
Belleza,292,9,3.08219178082192
Deportes,265,16,6.03773584905660
Hogar,275,17,6.18181818181818
Moda,311,7,2.25080385852090
Oficina,217,8,3.68663594470046
Tecnología,321,29,9.03426791277259


### 20. Ventas mensuales por vendedor y cumplimiento de meta

Calcula las ventas mensuales por vendedor y el porcentaje de cumplimiento de su meta mensual.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    v.vendedor_id,
    v.nombre_vendedor,
    DATE_FORMAT(p.fecha_pedido, 'yyyy-MM') AS mes,
    v.meta_mensual,
    SUM(dp.cantidad * dp.precio_unitario *
        (1 - dp.descuento_pct / 100.0)) AS ventas_mensuales,
    SUM(dp.cantidad * dp.precio_unitario *
        (1 - dp.descuento_pct / 100.0)) * 100.0 /
        v.meta_mensual AS porcentaje_meta
FROM vendedores v
JOIN pedidos p
    ON v.vendedor_id = p.vendedor_id
JOIN detalle_pedidos dp
    ON p.pedido_id = dp.pedido_id
WHERE p.estado <> 'cancelado'
GROUP BY
    v.vendedor_id,
    v.nombre_vendedor,
    DATE_FORMAT(p.fecha_pedido, 'yyyy-MM'),
    v.meta_mensual
ORDER BY mes, v.nombre_vendedor;

vendedor_id,nombre_vendedor,mes,meta_mensual,ventas_mensuales,porcentaje_meta
V009,Carlos Aguilar,2025-01,30000.00,1184.610000000,3.948700
V008,Diego Navarro,2025-01,12000.00,4725.544000000,39.379533
V001,José Cárdenas,2025-01,25000.00,5161.857500000,20.647430
V006,Luis Rojas,2025-01,12000.00,2034.420000000,16.953500
V005,Paola Silva,2025-01,30000.00,4255.136500000,14.183788
V010,Renzo Salazar,2025-01,18000.00,4388.987500000,24.383264
V012,Sebastián Herrera,2025-01,25000.00,8793.145000000,35.172580
V004,Camila Herrera,2025-02,18000.00,1972.505000000,10.958361
V009,Carlos Aguilar,2025-02,30000.00,192.908000000,0.643027
V008,Diego Navarro,2025-02,12000.00,9869.896500000,82.249138


### 21. Ranking de productos por ventas dentro de cada categoría

Genera un ranking de productos por ventas netas dentro de cada categoría.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    categoria,
    producto_id,
    nombre_producto,
    ventas_netas,
    ROW_NUMBER() OVER (
        PARTITION BY categoria
        ORDER BY ventas_netas DESC
    ) AS ranking
FROM (
    SELECT
        pr.categoria,
        pr.producto_id,
        pr.nombre_producto,
        SUM(dp.cantidad * dp.precio_unitario *
            (1 - dp.descuento_pct / 100.0)) AS ventas_netas
    FROM productos pr
    JOIN detalle_pedidos dp
        ON pr.producto_id = dp.producto_id
    JOIN pedidos p
        ON dp.pedido_id = p.pedido_id
    WHERE p.estado <> 'cancelado'
    GROUP BY pr.categoria, pr.producto_id, pr.nombre_producto
) AS ventas
ORDER BY categoria, ranking;

categoria,producto_id,nombre_producto,ventas_netas,ranking
Belleza,P0034,Crema Hidratante,6832.883000000,1
Belleza,P0035,Perfume Floral,6760.536000000,2
Belleza,P0030,Protector Solar,6104.948500000,3
Belleza,P0036,Shampoo Natural,5513.507000000,4
Belleza,P0031,Serum Facial,5154.619000000,5
Belleza,P0033,Kit de Brochas,3420.577000000,6
Belleza,P0032,Secadora Compacta,1732.885500000,7
Deportes,P0028,Guantes Fitness,11812.622000000,1
Deportes,P0023,Mat de Yoga,8294.421000000,2
Deportes,P0026,Banda Elástica,7300.201000000,3


### 22. Ventas mensuales y acumulado

Calcula las ventas netas mensuales y el acumulado de ventas desde el primer mes.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    mes,
    ventas_mensuales,
    SUM(ventas_mensuales) OVER (
        ORDER BY mes
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS ventas_acumuladas
FROM (
    SELECT
        DATE_FORMAT(p.fecha_pedido, 'yyyy-MM') AS mes,
        SUM(dp.cantidad * dp.precio_unitario *
            (1 - dp.descuento_pct / 100.0)) AS ventas_mensuales
    FROM pedidos p
    JOIN detalle_pedidos dp
        ON p.pedido_id = dp.pedido_id
    WHERE p.estado <> 'cancelado'
    GROUP BY DATE_FORMAT(p.fecha_pedido, 'yyyy-MM')
) AS ventas
ORDER BY mes;

mes,ventas_mensuales,ventas_acumuladas
2025-01,30543.700500000,30543.700500000
2025-02,37437.390500000,67981.091000000
2025-03,14321.272000000,82302.363000000
2025-04,29019.871000000,111322.234000000
2025-05,16668.402500000,127990.636500000
2025-06,21692.889000000,149683.525500000
2025-07,48426.429500000,198109.955000000
2025-08,45525.978500000,243635.933500000
2025-09,46879.126000000,290515.059500000
2025-10,52283.788000000,342798.847500000


### 23. Crecimiento porcentual respecto al mes anterior

Calcula el crecimiento porcentual de ventas netas respecto al mes anterior.

In [0]:
%sql
-- Escribe tu consulta aquí
WITH ventas AS (
    SELECT
        DATE_FORMAT(p.fecha_pedido, 'yyyy-MM') AS mes,
        SUM(dp.cantidad * dp.precio_unitario *
            (1 - dp.descuento_pct / 100.0)) AS ventas_mensuales
    FROM pedidos p
    JOIN detalle_pedidos dp
        ON p.pedido_id = dp.pedido_id
    WHERE p.estado <> 'cancelado'
    GROUP BY DATE_FORMAT(p.fecha_pedido, 'yyyy-MM')
),
comparacion AS (
    SELECT
        mes,
        ventas_mensuales,
        LAG(ventas_mensuales) OVER (ORDER BY mes) AS ventas_mes_anterior
    FROM ventas
)
SELECT
    mes,
    ventas_mensuales,
    ventas_mes_anterior,
    (ventas_mensuales - ventas_mes_anterior) * 100.0 /
        ventas_mes_anterior AS crecimiento_porcentual
FROM comparacion
ORDER BY mes;


mes,ventas_mensuales,ventas_mes_anterior,crecimiento_porcentual
2025-01,30543.700500000,null,null
2025-02,37437.390500000,30543.700500000,22.569924
2025-03,14321.272000000,37437.390500000,-61.746073
2025-04,29019.871000000,14321.272000000,102.634731
2025-05,16668.402500000,29019.871000000,-42.562107
2025-06,21692.889000000,16668.402500000,30.143780
2025-07,48426.429500000,21692.889000000,123.236423
2025-08,45525.978500000,48426.429500000,-5.989397
2025-09,46879.126000000,45525.978500000,2.972254
2025-10,52283.788000000,46879.126000000,11.528931


### 24. Primera y última compra por cliente

Para cada cliente con pedidos, muestra la fecha de su primera compra y la fecha de su compra más reciente.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    c.cliente_id,
    c.nombre_cliente,
    MIN(p.fecha_pedido) AS primera_compra,
    MAX(p.fecha_pedido) AS ultima_compra
FROM clientes c
JOIN pedidos p
    ON c.cliente_id = p.cliente_id
GROUP BY c.cliente_id, c.nombre_cliente
ORDER BY c.nombre_cliente;

cliente_id,nombre_cliente,primera_compra,ultima_compra
C0034,Alejandra Aguilar,2025-02-24,2026-01-20
C0112,Alejandra Campos,2025-04-16,2026-06-10
C0030,Alejandra Chávez,2025-08-20,2025-09-19
C0111,Alejandra Flores,2025-05-17,2025-12-19
C0094,Ana Campos,2025-04-07,2026-01-19
C0085,Ana Cruz,2025-07-31,2026-05-17
C0082,Ana Sánchez,2025-09-07,2026-01-24
C0049,Ana Torres,2025-10-24,2026-04-23
C0027,Ana Valdez,2026-01-28,2026-06-11
C0014,Andrea Medina,2025-11-09,2026-06-11


### 25. Clientes con compras válidas en al menos 3 meses distintos

Lista los clientes que realizaron pedidos válidos en al menos 3 meses distintos.

In [0]:
%sql
-- Escribe tu consulta aquí
SELECT
    c.cliente_id,
    c.nombre_cliente,
    COUNT(DISTINCT DATE_FORMAT(p.fecha_pedido, 'yyyy-MM')) AS meses_con_compra
FROM clientes c
JOIN pedidos p
    ON c.cliente_id = p.cliente_id
WHERE p.estado <> 'cancelado'
GROUP BY c.cliente_id, c.nombre_cliente
HAVING COUNT(DISTINCT DATE_FORMAT(p.fecha_pedido, 'yyyy-MM')) >= 3
ORDER BY meses_con_compra DESC;

cliente_id,nombre_cliente,meses_con_compra
C0090,Camila Medina,6
C0019,Daniela Peña,6
C0076,Mateo Vargas,5
C0065,Camila Castro,5
C0087,Carla Ramírez,5
C0045,Diego Valdez,5
C0034,Alejandra Aguilar,5
C0002,Natalia Aguilar,5
C0096,Gabriela Silva,5
C0057,Hugo Torres,5


### 26. Pedidos con importe superior al promedio

Muestra los pedidos válidos cuyo importe total sea superior al importe promedio de todos los pedidos válidos.

In [0]:
%sql
-- Escribe tu consulta aquí
WITH totales AS (
    SELECT
        p.pedido_id,
        SUM(dp.cantidad * dp.precio_unitario *
            (1 - dp.descuento_pct / 100.0)) AS importe_total
    FROM pedidos p
    JOIN detalle_pedidos dp
        ON p.pedido_id = dp.pedido_id
    WHERE p.estado <> 'cancelado'
    GROUP BY p.pedido_id
)
SELECT *
FROM totales
WHERE importe_total > (
    SELECT AVG(importe_total)
    FROM totales
)
ORDER BY importe_total DESC;

pedido_id,importe_total
O00156,14110.782500000
O00166,10814.875000000
O00259,10431.393000000
O00342,10417.274000000
O00096,10195.430000000
O00344,9540.529500000
O00012,9290.057500000
O00029,9285.176000000
O00150,9139.395000000
O00100,8932.322000000


### 27. Productos con precio superior al promedio de su categoría

Lista los productos cuyo precio de lista sea superior al precio promedio de su propia categoría.

In [0]:
%sql
-- Escribe tu consulta aquí

SELECT
    producto_id,
    nombre_producto,
    categoria,
    precio_lista
FROM productos p
WHERE precio_lista > (
    SELECT AVG(precio_lista)
    FROM productos
    WHERE categoria = p.categoria
)
ORDER BY categoria, precio_lista DESC;


producto_id,nombre_producto,categoria,precio_lista
P0033,Kit de Brochas,Belleza,185.00
P0035,Perfume Floral,Belleza,165.00
P0030,Protector Solar,Belleza,155.00
P0027,Botella Térmica,Deportes,255.00
P0026,Banda Elástica,Deportes,250.00
P0028,Guantes Fitness,Deportes,245.00
P0023,Mat de Yoga,Deportes,215.00
P0011,Freidora de Aire,Hogar,690.00
P0013,Organizador Modular,Hogar,685.00
P0014,Lámpara LED,Hogar,540.00


### 28. Participación de cada categoría en las ventas totales

Calcula el porcentaje de participación de cada categoría sobre las ventas netas válidas totales.

In [0]:
%sql
-- Escribe tu consulta aquí
WITH ventas AS (
    SELECT
        pr.categoria,
        SUM(dp.cantidad * dp.precio_unitario *
            (1 - dp.descuento_pct / 100.0)) AS ventas_netas
    FROM productos pr
    JOIN detalle_pedidos dp
        ON pr.producto_id = dp.producto_id
    JOIN pedidos p
        ON dp.pedido_id = p.pedido_id
    WHERE p.estado <> 'cancelado'
    GROUP BY pr.categoria
)
SELECT
    categoria,
    ventas_netas,
    ventas_netas * 100.0 /
        (SELECT SUM(ventas_netas) FROM ventas) AS porcentaje_total
FROM ventas
ORDER BY ventas_netas DESC;

categoria,ventas_netas,porcentaje_total
Tecnología,484789.875500000,58.823405
Oficina,125504.248000000,15.228427
Hogar,107762.248500000,13.075649
Deportes,46057.548500000,5.588528
Belleza,35519.956000000,4.309918
Moda,24510.648000000,2.974072


### 29. Método de pago más usado por ciudad de envío

Identifica el método de pago más utilizado en cada ciudad de envío. Si hay empate, elige alfabéticamente el método de pago.

In [0]:
%sql
-- Escribe tu consulta aquí

WITH conteos AS (
    SELECT
        ciudad_envio,
        metodo_pago,
        COUNT(*) AS cantidad_usos
    FROM pedidos
    GROUP BY ciudad_envio, metodo_pago
),
ranking AS (
    SELECT
        ciudad_envio,
        metodo_pago,
        cantidad_usos,
        ROW_NUMBER() OVER (
            PARTITION BY ciudad_envio
            ORDER BY cantidad_usos DESC, metodo_pago ASC
        ) AS posicion
    FROM conteos
)
SELECT
    ciudad_envio,
    metodo_pago,
    cantidad_usos
FROM ranking
WHERE posicion = 1
ORDER BY ciudad_envio;


ciudad_envio,metodo_pago,cantidad_usos
Arequipa,Tarjeta,9
Ayacucho,Yape,8
Cajamarca,Tarjeta,14
Chiclayo,Yape,10
Cusco,Tarjeta,10
Huancayo,Tarjeta,8
Ica,Tarjeta,8
Lima,Tarjeta,14
Piura,Tarjeta,13
Puno,Tarjeta,12


### 30. Días promedio entre registro y primera compra por segmento

Calcula, por segmento de cliente, el promedio de días transcurridos entre la fecha de registro y la primera compra.

In [0]:
%sql
-- Escribe tu consulta aquí
WITH primeras_compras AS (
    SELECT
        cliente_id,
        MIN(fecha_pedido) AS primera_compra
    FROM pedidos
    GROUP BY cliente_id
)
SELECT
    c.segmento,
    AVG(DATEDIFF(f.primera_compra, c.fecha_registro)) AS promedio_dias
FROM clientes c
JOIN primeras_compras f
    ON c.cliente_id = f.cliente_id
GROUP BY c.segmento
ORDER BY c.segmento;

segmento,promedio_dias
Empresa,188.0
Persona,185.73972602739727
Pyme,232.9
